TASK1

In [1]:
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import duckdb
import os
import time
from datetime import datetime, timedelta

In [2]:
np.random.seed(42)

n_rows = 500_000
cities = ["Berlin", "Tokyo", "New York", "London", "Paris", 
          "Sydney", "Moscow", "Toronto", "Dubai", "Beijing"]

df = pd.DataFrame({
    "user_id": np.arange(1, n_rows+1),
    "city": np.random.choice(cities, size=n_rows),
    "score": np.random.uniform(0, 100, size=n_rows),
    "active": np.random.choice([True, False], size=n_rows),
    "signup_date": pd.to_datetime('2023-03-06') - pd.to_timedelta(np.random.randint(0, 3*365, size=n_rows), unit='D'),
    "age": np.random.randint(18, 81, size=n_rows),
    "sessions": np.random.randint(0, 501, size=n_rows),
    "revenue": np.random.uniform(0, 1000, size=n_rows)
})

In [3]:
parquet_file = "users.parquet"
df.to_parquet(parquet_file, engine="pyarrow", index=False)

pf = pq.ParquetFile(parquet_file)
print("Number of row groups:", pf.num_row_groups)
print("Number of rows:", pf.metadata.num_rows)
print("Number of columns:", pf.metadata.num_columns)
print("Schema:")
print(pf.schema)

Number of row groups: 1
Number of rows: 500000
Number of columns: 8
Schema:
required group field_id=-1 schema {
  optional int64 field_id=-1 user_id;
  optional binary field_id=-1 city (String);
  optional double field_id=-1 score;
  optional boolean field_id=-1 active;
  optional int64 field_id=-1 signup_date (Timestamp(isAdjustedToUTC=false, timeUnit=microseconds, is_from_converted_type=false, force_set_converted_type=false));
  optional int32 field_id=-1 age;
  optional int32 field_id=-1 sessions;
  optional double field_id=-1 revenue;
}



In [4]:
print("\nFirst row group column metadata:")
for i in range(pf.metadata.num_columns):
    col = pf.metadata.row_group(0).column(i)
    print(f"{col.path_in_schema}: "
          f"type={col.physical_type}, "
          f"compressed_size={col.total_compressed_size}, " 
          f"min={col.statistics.min if col.statistics else None}, "
          f"max={col.statistics.max if col.statistics else None}, "
          f"null_count={col.statistics.null_count if col.statistics else None}")


First row group column metadata:
user_id: type=INT64, compressed_size=2273849, min=1, max=500000, null_count=0
city: type=BYTE_ARRAY, compressed_size=252639, min=Beijing, max=Toronto, null_count=0
score: type=DOUBLE, compressed_size=4272959, min=5.188445665327279e-05, max=99.99983148609545, null_count=0
active: type=BOOLEAN, compressed_size=63800, min=False, max=True, null_count=0
signup_date: type=INT64, compressed_size=698254, min=2020-03-07 00:00:00, max=2023-03-06 00:00:00, null_count=0
age: type=INT32, compressed_size=377947, min=18, max=80, null_count=0
sessions: type=INT32, compressed_size=567227, min=0, max=500, null_count=0
revenue: type=DOUBLE, compressed_size=4272959, min=0.0009958058022618843, max=999.9978869129644, null_count=0


In [5]:
csv_file = "users.csv"
df.to_csv(csv_file, index=False)

In [6]:
parquet_size = os.path.getsize(parquet_file)/1024
csv_size = os.path.getsize(csv_file)/1024
compression_ratio = csv_size / parquet_size
print(f"Parquet size: {parquet_size:.2f} KB")
print(f"CSV size: {csv_size:.2f} KB")
print(f"Compression ratio (CSV/Parquet): {compression_ratio:.2f}")

Parquet size: 12484.33 KB
CSV size: 36379.03 KB
Compression ratio (CSV/Parquet): 2.91


Parquet metadata stores detailed statistics like min/max values, null counts, and data types for every column chunk, whereas CSV is a schema-less text format. This metadata allows query engines to skip irrelevant data blocks entirely (block skipping) and perform type-safe operations without scanning the whole file.

TASK2

In [7]:
%time pq.read_table(parquet_file).to_pandas()

CPU times: total: 93.8 ms
Wall time: 66.1 ms


,user_id,city,score,active,signup_date,age,sessions,revenue
0,1,Moscow,43.689490,False,2023-02-02,71,323,179.164861
1,2,London,97.403462,True,2022-06-03,39,352,614.893856
2,3,Toronto,66.148238,False,2022-11-09,23,279,217.069799
3,4,Paris,21.993544,False,2022-05-23,20,129,195.459273
4,5,Moscow,91.719953,True,2020-04-02,67,199,617.349590
...,...,...,...,...,...,...,...,...
499995,499996,London,61.531318,False,2022-04-20,67,448,157.442594
499996,499997,Berlin,74.712924,False,2022-10-26,74,36,418.119068
499997,499998,Dubai,48.430697,True,2020-09-03,27,205,803.122069
499998,499999,Toronto,89.680757,True,2023-01-25,50,432,602.050097


In [8]:
%time pq.read_table(parquet_file, columns=["user_id", "score"]).to_pandas()

CPU times: total: 15.6 ms
Wall time: 19.8 ms


,user_id,score
0,1,43.689490
1,2,97.403462
2,3,66.148238
3,4,21.993544
4,5,91.719953
...,...,...
499995,499996,61.531318
499996,499997,74.712924
499997,499998,48.430697
499998,499999,89.680757


In [9]:
%time pd.read_csv(csv_file, usecols=["user_id", "score"])

CPU times: total: 484 ms
Wall time: 477 ms


,user_id,score
0,1,43.689490
1,2,97.403462
2,3,66.148238
3,4,21.993544
4,5,91.719953
...,...,...
499995,499996,61.531318
499996,499997,74.712924
499997,499998,48.430697
499998,499999,89.680757


Parquet’s columnar storage layout allows the reader to seek directly to the byte offsets of specific columns, skipping the data for all other columns on disk. In contrast, CSV is row-based, requiring the computer to read every character of every column in a row just to find the specific fields you need.

TASK3

In [10]:
%%time 
table_filtered = pq.read_table(parquet_file, filters=[('age', '>', 50)])
df_filtered_arrow = table_filtered.to_pandas()

CPU times: total: 109 ms
Wall time: 52.2 ms


In [11]:
%%time
df_full = pd.read_parquet(parquet_file)
df_filtered_post = df_full[df_full['age'] > 50]

CPU times: total: 78.1 ms
Wall time: 62.4 ms


Predicate pushdown filters data at the storage layer by using row group statistics (like min/max) to determine if a block even contains the requested values before loading it into memory. This drastically reduces I/O and memory usage compared to loading the entire dataset and filtering it in a high-level library like pandas.

In [12]:
%time result_duck = duckdb.sql(f"SELECT * FROM '{parquet_file}' WHERE age > 50").df()

CPU times: total: 109 ms
Wall time: 111 ms


TASK4

In [13]:
%time df.groupby('city')['user_id'].count()

CPU times: total: 15.6 ms
Wall time: 25.9 ms


city
Beijing     50061
Berlin      50059
Dubai       49724
London      49931
Moscow      49982
New York    50113
Paris       50216
Sydney      50059
Tokyo       49970
Toronto     49885
Name: user_id, dtype: int64

In [14]:
%time duckdb.sql(f"SELECT city, COUNT(user_id) FROM '{parquet_file}' GROUP BY city").df()

CPU times: total: 15.6 ms
Wall time: 26.4 ms


,city,count(user_id)
0,Moscow,49982
1,Sydney,50059
2,Berlin,50059
3,Dubai,49724
4,Paris,50216
5,New York,50113
6,Beijing,50061
7,Tokyo,49970
8,London,49931
9,Toronto,49885


In [15]:
%time df.groupby('city')['score'].mean().sort_values(ascending=False)

CPU times: total: 46.9 ms
Wall time: 29.7 ms


city
Sydney      50.311448
Paris       50.151899
Tokyo       50.135854
Moscow      50.075511
Toronto     50.063780
Berlin      50.013372
London      50.009568
Dubai       49.996347
Beijing     49.994487
New York    49.879502
Name: score, dtype: float64

In [16]:
%time duckdb.sql(f"SELECT city, AVG(score) as avg_score FROM '{parquet_file}' GROUP BY city ORDER BY avg_score DESC").df()

CPU times: total: 15.6 ms
Wall time: 22.9 ms


,city,avg_score
0,Sydney,50.311448
1,Paris,50.151899
2,Tokyo,50.135854
3,Moscow,50.075511
4,Toronto,50.063780
5,Berlin,50.013372
6,London,50.009568
7,Dubai,49.996347
8,Beijing,49.994487
9,New York,49.879502


In [17]:
%time df_active = df[df['score'] > 75].groupby('city')['active'].mean()

CPU times: total: 15.6 ms
Wall time: 22.4 ms


In [18]:
%%time
duckdb.sql(f"""
SELECT city, 100.0*AVG(CASE WHEN active THEN 1 ELSE 0 END) AS pct_active_high
FROM '{parquet_file}' WHERE score > 75 GROUP BY city
""").df()

CPU times: total: 31.2 ms
Wall time: 29.6 ms


,city,pct_active_high
0,New York,49.444044
1,Dubai,49.695415
2,Paris,49.717739
3,London,50.466396
4,Toronto,50.084303
5,Beijing,50.152586
6,Tokyo,50.479004
7,Moscow,50.067691
8,Sydney,50.051193
9,Berlin,50.056126


In [19]:
%%time
df['rank'] = df.groupby('city')['score'].rank(method='first', ascending=False)
df_top10 = df[df['rank']<=10]

CPU times: total: 453 ms
Wall time: 454 ms


In [20]:
%%time
duckdb.sql(f"""
SELECT * FROM (
    SELECT *, RANK() OVER (PARTITION BY city ORDER BY score DESC) as rnk
    FROM '{parquet_file}'
) WHERE rnk <= 10
""").df()

CPU times: total: 969 ms
Wall time: 450 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,rnk
0,83418,Tokyo,99.999831,True,2022-09-22,67,356,284.676962,1
1,332637,Tokyo,99.996193,True,2023-02-09,52,268,629.844270,2
2,184308,Tokyo,99.993404,True,2021-12-08,48,418,192.964934,3
3,322632,Tokyo,99.993183,True,2022-03-03,69,482,598.988574,4
4,165102,Tokyo,99.993058,False,2021-08-31,71,491,91.484437,5
...,...,...,...,...,...,...,...,...,...
95,343772,Dubai,99.991486,True,2022-01-03,41,471,944.117265,6
96,115967,Dubai,99.989740,True,2020-04-10,40,261,955.513281,7
97,78469,Dubai,99.987544,True,2021-09-24,38,278,714.991561,8
98,12514,Dubai,99.986537,True,2020-12-30,69,461,313.148120,9


In [21]:
df['running_total'] = df.groupby('city')['score'].cumsum()

In [22]:
%%time
duckdb.sql(f"""
SELECT *, SUM(score) OVER (PARTITION BY city ORDER BY user_id) AS running_total
FROM '{parquet_file}'
""").df()

CPU times: total: 594 ms
Wall time: 400 ms


,user_id,city,score,active,signup_date,age,sessions,revenue,running_total
0,115613,Toronto,18.331533,True,2020-07-10,46,4,281.486199,5.799131e+05
1,115624,Toronto,39.185892,False,2021-09-01,60,234,879.550907,5.799523e+05
2,115637,Toronto,63.652909,False,2020-07-22,51,194,147.840051,5.800160e+05
3,115643,Toronto,61.039900,True,2023-01-04,71,362,971.027347,5.800770e+05
4,115647,Toronto,97.067800,False,2022-08-05,52,15,143.260780,5.801741e+05
...,...,...,...,...,...,...,...,...,...
499995,409154,New York,11.566792,True,2020-05-14,49,262,898.059033,2.042485e+06
499996,409166,New York,59.964096,False,2020-07-17,42,488,149.610958,2.042545e+06
499997,409180,New York,46.827287,True,2021-12-05,49,410,70.154686,2.042592e+06
499998,409194,New York,49.525899,False,2020-06-07,44,413,518.851120,2.042642e+06


DuckDB is generally faster for analytical queries because its vectorized engine processes batches of data optimized for CPU caches, whereas pandas often performs slower, row-wise or object-heavy operations. While pandas is more flexible for procedural data manipulation, DuckDB’s SQL is more concise for complex aggregations and window functions.

TASK5

In [23]:
table = pa.Table.from_pandas(df)

In [24]:
table.schema

user_id: int64
city: large_string
score: double
active: bool
signup_date: timestamp[us]
age: int32
sessions: int32
revenue: double
rank: double
running_total: double
-- schema metadata --
pandas: '{"index_columns": [{"kind": "range", "name": null, "start": 0, "' + 1410

In [25]:
df.dtypes

user_id                   int64
city                        str
score                   float64
active                     bool
signup_date      datetime64[us]
age                       int32
sessions                  int32
revenue                 float64
rank                    float64
running_total           float64
dtype: object

In [26]:
pq.write_table(table, "arrow_table.parquet")
table_back = pq.read_table("arrow_table.parquet")

In [27]:
df_back = table_back.to_pandas()

In [28]:
assert df.equals(df_back), 'Not equal'

In [29]:
df_arrow = pd.read_parquet('arrow_table.parquet', dtype_backend='pyarrow')

In [30]:
df_arrow.dtypes

user_id                  int64[pyarrow]
city              large_string[pyarrow]
score                   double[pyarrow]
active                    bool[pyarrow]
signup_date      timestamp[us][pyarrow]
age                      int32[pyarrow]
sessions                 int32[pyarrow]
revenue                 double[pyarrow]
rank                    double[pyarrow]
running_total           double[pyarrow]
dtype: object

Apache Arrow acts as a standardized, in-memory columnar format that allows data to move between Parquet (disk), DuckDB (engine), and pandas (analysis) without expensive serialization or conversion. It serves as the "common language" that enables high-performance interoperability across the modern data stack.